# 🇬🇧 UK Retail Performance Hub
## Phase 1 | Notebook 4 of 4 — Automated Refresh & GitHub Push

**What this notebook does:**
- Runs the full pipeline in one go (ingest → clean → load → export)
- Logs every run with timestamps
- Pushes updated outputs back to your GitHub repo automatically

**This is the notebook you'd run when you want to refresh everything.**

> ℹ️ In a real job, this would be scheduled to run daily via a cron job or Azure Data Factory.
> In Colab, you run it manually — but the logging and GitHub push show the same professional approach.

---

### Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ROOT     = Path('/content/drive/MyDrive/uk-retail-performance-hub')
LOG_DIR  = ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f'✓ Drive mounted')
print(f'  Root: {ROOT}')

### Cell 2 — Connect to GitHub
Clones your repo into Colab so we can push files back to it at the end.

> **Before running:** Replace `YOUR_GITHUB_TOKEN` with a Personal Access Token from GitHub.
> Get one at: GitHub → Settings → Developer Settings → Personal Access Tokens → Generate new token
> Only needs `repo` scope ticked.

In [ ]:
import os

# ── YOUR GITHUB DETAILS — fill these in ───────────────────────────────────────
GITHUB_USERNAME = 'Prati2707'
GITHUB_REPO     = 'uk-retail-performance-hub'
GITHUB_TOKEN    = 'YOUR_GITHUB_TOKEN'   # Replace with your token
GITHUB_EMAIL    = 'your@email.com'      # Replace with your GitHub email
# ─────────────────────────────────────────────────────────────────────────────

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
REPO_DIR = Path(f'/content/{GITHUB_REPO}')

# Clone if not already cloned this session
if REPO_DIR.exists():
    print(f'✓ Repo already cloned at {REPO_DIR}')
else:
    !git clone {REPO_URL} {REPO_DIR}
    print(f'✓ Repo cloned to {REPO_DIR}')

# Configure git identity
!git -C {REPO_DIR} config user.email "{GITHUB_EMAIL}"
!git -C {REPO_DIR} config user.name "{GITHUB_USERNAME}"
print('✓ Git configured')

### Cell 3 — Set up logging

In [ ]:
import logging
from datetime import datetime

LOG_FILE = LOG_DIR / 'pipeline_run.log'

# Set up logging to both file and notebook output
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)
log.info('Pipeline logger initialised')
print(f'✓ Logs will be written to {LOG_FILE}')

### Cell 4 — Run full pipeline
This runs all three processing steps in sequence with timing and logging.
**This cell will take 3–5 minutes to complete.**

In [ ]:
import sqlite3
import requests
import pandas as pd
import numpy as np
import time

start_time = datetime.now()
RAW_DIR       = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
POWERBI_DIR   = ROOT / 'data' / 'powerbi'

for d in [RAW_DIR, PROCESSED_DIR, POWERBI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = PROCESSED_DIR / 'retail_hub.db'

log.info('=' * 55)
log.info('PIPELINE REFRESH STARTED')
log.info('=' * 55)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 1: INGESTION
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
log.info('STEP 1: Ingestion')
step_start = time.time()

uci_path = RAW_DIR / 'online_retail.xlsx'
if uci_path.exists():
    log.info(f'  UCI file already present — skipping download')
else:
    log.info('  Downloading UCI dataset...')
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
    try:
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        uci_path.write_bytes(r.content)
        log.info(f'  UCI downloaded ({uci_path.stat().st_size/1e6:.1f} MB)')
    except Exception as e:
        log.error(f'  UCI download failed: {e}')
        raise

log.info(f'  Step 1 complete ({time.time()-step_start:.0f}s)')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 2: CLEANING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
log.info('STEP 2: Cleaning')
step_start = time.time()

log.info('  Loading Excel file...')
df = pd.read_excel(uci_path, dtype={'CustomerID': str})
log.info(f'  Raw rows: {len(df):,}')

df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[df['CustomerID'].notna()]
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df = df[~df['StockCode'].astype(str).str.fullmatch(r'[A-Za-z]+')]
log.info(f'  Clean rows: {len(df):,}')

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Revenue']     = (df['Quantity'] * df['UnitPrice']).round(2)
df['Year']        = df['InvoiceDate'].dt.year
df['Month']       = df['InvoiceDate'].dt.month
df['YearMonth']   = df['InvoiceDate'].dt.to_period('M').astype(str)
df['Quarter']     = df['InvoiceDate'].dt.to_period('Q').astype(str)
df['DayOfWeek']   = df['InvoiceDate'].dt.day_name()
df['IsWeekend']   = df['InvoiceDate'].dt.dayofweek >= 5
df['IsUK']        = df['Country'].str.strip().str.upper() == 'UNITED KINGDOM'

first_purchase = (
    df.groupby('CustomerID')['InvoiceDate'].min()
    .dt.to_period('M').astype(str).rename('CohortMonth')
)
df = df.merge(first_purchase, on='CustomerID', how='left')

CATEGORY_KEYWORDS = {
    'Christmas & Seasonal'  : ['christmas','xmas','santa','reindeer','advent','tinsel','bauble','wreath','snowflake'],
    'Candles & Lighting'    : ['candle','lantern','tealight','tea light','nightlight','lamp','light'],
    'Bags & Accessories'    : ['bag','tote','purse','wallet','satchel','handbag','pouch'],
    'Home Décor'            : ['frame','mirror','clock','wall','door','cushion','bunting','sign','plaque'],
    'Kitchenware & Dining'  : ['mug','cup','bowl','plate','tin','jar','jug','bottle','spoon','napkin','coaster','tray'],
    'Stationery & Cards'    : ['card','notebook','diary','pen','pencil','sticker','label','wrap','ribbon','tag'],
    'Toys & Games'          : ['toy','game','doll','ball','puppet','puzzle','play'],
    'Novelty & Gifts'       : ['gift','novel','fun','party','balloon','charm','badge'],
}

def categorise(desc):
    if not isinstance(desc, str): return 'Uncategorised'
    d = desc.lower()
    for cat, kws in CATEGORY_KEYWORDS.items():
        if any(k in d for k in kws): return cat
    return 'Uncategorised'

df['ProductCategory'] = df['Description'].apply(categorise)
df['IsUK']      = df['IsUK'].astype(int)
df['IsWeekend'] = df['IsWeekend'].astype(int)

df.to_parquet(PROCESSED_DIR / 'uci_clean.parquet', index=False)
log.info(f'  UCI clean saved. Step 2 complete ({time.time()-step_start:.0f}s)')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 3: LOAD TO DATABASE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
log.info('STEP 3: Load to database')
step_start = time.time()

conn = sqlite3.connect(DB_PATH)
df_copy = df.copy()
df_copy['InvoiceDate'] = df_copy['InvoiceDate'].astype(str)
df_copy.to_sql('transactions', conn, if_exists='replace', index=False)

# Create views
cursor = conn.cursor()
view_sqls = [
    ('vw_monthly_revenue', '''
        CREATE VIEW IF NOT EXISTS vw_monthly_revenue AS
        SELECT YearMonth, Year, Month,
            COUNT(DISTINCT InvoiceNo)  AS OrderCount,
            COUNT(DISTINCT CustomerID) AS ActiveCustomers,
            ROUND(SUM(Revenue),2)      AS TotalRevenue,
            SUM(Quantity)              AS UnitsSold,
            ROUND(SUM(CASE WHEN IsUK=1 THEN Revenue ELSE 0 END),2) AS UKRevenue,
            ROUND(SUM(CASE WHEN IsUK=0 THEN Revenue ELSE 0 END),2) AS IntlRevenue
        FROM transactions GROUP BY YearMonth,Year,Month ORDER BY YearMonth'''),
    ('vw_customer_summary', '''
        CREATE VIEW IF NOT EXISTS vw_customer_summary AS
        SELECT CustomerID, Country, IsUK, CohortMonth,
            COUNT(DISTINCT InvoiceNo) AS TotalOrders,
            ROUND(SUM(Revenue),2) AS TotalRevenue,
            ROUND(AVG(Revenue),2) AS AvgOrderValue,
            MIN(InvoiceDate) AS FirstPurchase, MAX(InvoiceDate) AS LastPurchase,
            CAST(JULIANDAY('2011-12-09') - JULIANDAY(MAX(InvoiceDate)) AS INTEGER) AS DaysSinceLastPurchase,
            COUNT(DISTINCT YearMonth) AS ActiveMonths
        FROM transactions GROUP BY CustomerID,Country,IsUK,CohortMonth'''),
    ('vw_product_performance', '''
        CREATE VIEW IF NOT EXISTS vw_product_performance AS
        SELECT ProductCategory, StockCode, Description,
            COUNT(DISTINCT InvoiceNo) AS OrderCount, SUM(Quantity) AS UnitsSold,
            ROUND(SUM(Revenue),2) AS TotalRevenue, ROUND(AVG(UnitPrice),2) AS AvgUnitPrice,
            COUNT(DISTINCT CustomerID) AS UniqueCustomers
        FROM transactions GROUP BY ProductCategory,StockCode,Description ORDER BY TotalRevenue DESC'''),
    ('vw_cohort_retention', '''
        CREATE VIEW IF NOT EXISTS vw_cohort_retention AS
        SELECT CohortMonth, YearMonth AS ActivityMonth,
            COUNT(DISTINCT CustomerID) AS ActiveCustomers,
            CAST((SUBSTR(YearMonth,1,4)*12+CAST(SUBSTR(YearMonth,6,2) AS INT))
                -(SUBSTR(CohortMonth,1,4)*12+CAST(SUBSTR(CohortMonth,6,2) AS INT)) AS INTEGER) AS MonthOffset
        FROM transactions GROUP BY CohortMonth,YearMonth ORDER BY CohortMonth,YearMonth'''),
]

for view_name, sql in view_sqls:
    cursor.execute(f'DROP VIEW IF EXISTS {view_name}')
    cursor.execute(sql)
conn.commit()

# Export to CSV
for view_name in ['vw_monthly_revenue','vw_customer_summary','vw_product_performance','vw_cohort_retention']:
    fname = view_name.replace('vw_','') + '.csv'
    pd.read_sql(f'SELECT * FROM {view_name}', conn).to_csv(POWERBI_DIR / fname, index=False)

conn.close()
log.info(f'  Database and CSVs updated. Step 3 complete ({time.time()-step_start:.0f}s)')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FINAL SUMMARY
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
duration = (datetime.now() - start_time).seconds
log.info('=' * 55)
log.info(f'PIPELINE COMPLETE in {duration}s')
log.info(f'  Rows processed : {len(df):,}')
log.info(f'  Total revenue  : £{df["Revenue"].sum():,.2f}')
log.info('=' * 55)

### Cell 5 — Push outputs to GitHub
Copies the Power BI CSVs and log file to your repo and pushes to GitHub.

This is the part that shows recruiters you understand professional data workflows — auto-committing outputs with timestamped run logs.

In [ ]:
import shutil

# Copy Power BI CSVs into the repo
repo_data_dir = REPO_DIR / 'data' / 'powerbi'
repo_data_dir.mkdir(parents=True, exist_ok=True)

for csv_file in POWERBI_DIR.glob('*.csv'):
    shutil.copy(csv_file, repo_data_dir / csv_file.name)
    print(f'  Copied: {csv_file.name}')

# Copy log file
repo_log_dir = REPO_DIR / 'logs'
repo_log_dir.mkdir(exist_ok=True)
shutil.copy(LOG_FILE, repo_log_dir / 'pipeline_run.log')
print(f'  Copied: pipeline_run.log')

# Git commit and push
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M')
commit_msg = f'Automated refresh: {timestamp}'

!git -C {REPO_DIR} add data/powerbi/ logs/
!git -C {REPO_DIR} commit -m "{commit_msg}"
!git -C {REPO_DIR} push

print(f'\n✓ Pushed to GitHub with message: "{commit_msg}"')
print(f'  View at: https://github.com/Prati2707/uk-retail-performance-hub')

### Cell 6 — View the run log
Shows you the last 30 lines of the log file — useful to share in your portfolio or show in an interview.

In [ ]:
print('PIPELINE RUN LOG')
print('=' * 55)
with open(LOG_FILE, 'r') as f:
    lines = f.readlines()

# Print last 30 lines
for line in lines[-30:]:
    print(line.rstrip())